# Референсная модель: 2025 как эталонный год

**Идея:** берём квартальный профиль 2025 как шаблон сезонности, масштабируем на 2026 по двум сигналам:
- **g_hist** — исторический тренд: sum(2025) / sum(2024)
- **g_Q1** — сигнал фактического начала года: Q1-2026 / Q1-2025 (cap 1.5)

**Три сценария:**
- Консервативный: g = g_hist (доверяем только истории)
- Базовый: g = (g_hist + g_Q1) / 2 (взвешенный сигнал)
- Оптимистичный: g = g_Q1 (доверяем Q1 2026)

**Доверительный коридор** = полуразброс между консервативным и оптимистичным сценариями.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 130

quarters = [
    'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024',
    'Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025',
    'Q1 2026',
]
raw = {
    'Выступления':         [1595, 8913, 16406, 27472, 9780, 18358, 10499, 28077, 12350],
    'Платные выступления': [1005, 5615, 10336, 17307, 5281,  9913,  5669, 15162,  5070],
    'Оплаты':              [ 365, 2384,  2970,  9668, 1077,  5548,  5212, 10210,  3761],
}

n_hist            = len(quarters)
forecast_quarters = ['Q2 2026', 'Q3 2026', 'Q4 2026']
all_quarters      = quarters + forecast_quarters

print('Данные загружены.')
print(pd.DataFrame(raw, index=quarters).to_string())

In [ ]:
# ─── Референсная модель ──────────────────────────────────────────────────────
G_Q1_CAP = 1.5   # cap на Q1-сигнал: не доверяем аномальному росту > 1.5x

results = {}
for metric, y in raw.items():
    y24   = np.array(y[0:4], float)
    y25   = np.array(y[4:8], float)
    q1_26 = float(y[8])

    g_hist = sum(y25) / sum(y24)           # годовой тренд 2024→2025
    g_q1   = q1_26 / y25[0]               # Q1 2026 vs Q1 2025
    g_q1c  = min(g_q1, G_Q1_CAP)          # с cap

    g_cons = g_hist
    g_base = (g_hist + g_q1c) / 2
    g_opt  = g_q1c

    # Прогноз = Q2..Q4 2025 * g_scenario
    ref_vals = y25[1:]   # [Q2, Q3, Q4] 2025
    fc = {
        'Консервативный': ref_vals * g_cons,
        'Базовый':        ref_vals * g_base,
        'Оптимистичный':  ref_vals * g_opt,
    }

    # CI = (оптимистичный - консервативный) / 2
    ci = np.abs(fc['Оптимистичный'] - fc['Консервативный']) / 2

    results[metric] = {
        'y24': y24, 'y25': y25, 'q1_26': q1_26,
        'g_hist': g_hist, 'g_q1': g_q1, 'g_q1c': g_q1c,
        'fc': fc, 'ci': ci,
    }

print('Модель рассчитана.\n')
print(f'{"Метрика":<25}  g_hist  g_Q1_raw  g_Q1_cap')
print('-' * 55)
for m, res in results.items():
    cap_note = f' (cap {res["g_q1c"]:.2f})' if res['g_q1'] > G_Q1_CAP else ''
    print(f'  {m:<25} {res["g_hist"]:.2f}    {res["g_q1"]:.2f}      {res["g_q1c"]:.2f}{cap_note}')

In [ ]:
# ─── Итоговые таблицы ────────────────────────────────────────────────────────
print('=' * 80)
print('РЕФЕРЕНСНАЯ МОДЕЛЬ (2025 = эталон): все сценарии')
print('=' * 80)
print(f'{"Метрика":<25} {"Сценарий":<18} {"Q2":>9} {"Q3":>9} {"Q4":>9}')
print('-' * 80)
for metric, res in results.items():
    for name in ['Консервативный', 'Базовый', 'Оптимистичный']:
        fc = res['fc'][name]
        row = '  '.join(f'{int(round(v)):>9,}'.replace(',', ' ') for v in fc)
        marker = ' *' if name == 'Базовый' else ''
        print(f'{metric:<25} {name:<18} {row}{marker}')
    print()

print('\n=== Базовый сценарий + доверительный коридор ===')
print(f'{"Метрика":<25} {"Квартал":<10} {"Прогноз":>9} {"Мин":>9} {"Макс":>9}')
print('-' * 70)
for metric, res in results.items():
    for i, q in enumerate(forecast_quarters):
        v  = res['fc']['Базовый'][i]
        lo = max(0, res['fc']['Консервативный'][i])
        hi = res['fc']['Оптимистичный'][i]
        print(f"{metric:<25} {q:<10} {int(round(v)):>9,} {int(round(lo)):>9,} {int(round(hi)):>9,}".replace(',', ' '))

In [ ]:
# ─── Визуализация ────────────────────────────────────────────────────────────
metric_colors = {
    'Выступления':         '#2563EB',
    'Платные выступления': '#16A34A',
    'Оплаты':              '#DC2626',
}

fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)
fig.suptitle(
    'Референсная модель: 2025 как эталон\nПрогноз Q2–Q4 2026',
    fontsize=15, fontweight='bold', y=0.999
)

x_all  = np.arange(len(all_quarters))
x_hist = x_all[:n_hist]
x_fc   = x_all[n_hist:]

for ax, (metric, res) in zip(axes, results.items()):
    mc = metric_colors[metric]
    y_full = list(res['y24']) + list(res['y25']) + [res['q1_26']]

    # История
    ax.scatter(x_hist, y_full, color=mc, s=70, zorder=6)
    ax.plot(x_hist, y_full, '-', color=mc, lw=1.3, alpha=0.5)

    # Эталонный 2025
    ax.plot(x_all[4:8], res['y25'], 's', color='#6B7280', ms=7,
            alpha=0.6, zorder=4, label='2025 (эталон)')
    ax.plot(x_all[4:8], res['y25'], '--', color='#6B7280', lw=1.2, alpha=0.4)

    base  = res['fc']['Базовый']
    cons  = res['fc']['Консервативный']
    opt   = res['fc']['Оптимистичный']

    # Коридор (консерв.–оптимист.)
    ax.fill_between(x_fc, np.maximum(0, cons), opt,
                    color=mc, alpha=0.15, label='Коридор (конс.–опт.)')

    # Три линии
    ax.plot(x_fc, cons, 'o--', color=mc, lw=1.5, ms=6, alpha=0.7, label='Консервативный')
    ax.plot(x_fc, opt,  'o-.', color=mc, lw=1.5, ms=6, alpha=0.7, label='Оптимистичный')
    ax.plot(x_fc, base, 'o-',  color=mc, lw=2.8, ms=9, zorder=5,  label='Базовый')

    # Подписи базового
    for xi, yi in zip(x_fc, base):
        ax.annotate(f'{int(round(yi)):,}'.replace(',', '\u202f'),
                    xy=(xi, yi), xytext=(0, 11), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold', color=mc)

    ax.axvline(x=n_hist - 0.5, color='gray', ls=':', lw=1.5)
    ylo, yhi = ax.get_ylim()
    ax.text(n_hist - 0.38, yhi * 0.97, 'прогноз ->', fontsize=8, color='gray', va='top')

    cap_note = f', Q1-cap={res["g_q1c"]:.2f}' if res['g_q1'] > G_Q1_CAP else ''
    ax.set_title(
        f'{metric}   [g_hist={res["g_hist"]:.2f}  |  g_Q1={res["g_q1"]:.2f}{cap_note}]',
        fontsize=11, fontweight='bold', pad=6
    )
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', '\u202f'))
    )
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper left', fontsize=8.5, framealpha=0.9, ncol=3)

axes[-1].set_xticks(x_all)
axes[-1].set_xticklabels(all_quarters, rotation=35, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('forecast_2026_reference.png', bbox_inches='tight', dpi=150)
plt.show()
print('График сохранён: forecast_2026_reference.png')

In [ ]:
# ─── Сравнение с ансамблевой моделью ─────────────────────────────────────────
ensemble_fc = {
    'Выступления':         [23008, 15511, 32383],
    'Платные выступления': [10630,  7381, 16279],
    'Оплаты':              [13415, 11052, 12291],
}

print('=' * 85)
print('СРАВНЕНИЕ: Референсная (базовый) vs Ансамблевая')
print('=' * 85)
print(f'{"Метрика":<25} {"Модель":<22} {"Q2":>9} {"Q3":>9} {"Q4":>9}')
print('-' * 85)
for metric, res in results.items():
    ref_fc = res['fc']['Базовый']
    ens_fc = ensemble_fc[metric]
    print(f'{metric:<25} {"Референсная (баз.)":<22} '
          + '  '.join(f'{int(round(v)):>9,}'.replace(',', ' ') for v in ref_fc))
    print(f'{"":25} {"Ансамблевая":<22} '
          + '  '.join(f'{int(round(v)):>9,}'.replace(',', ' ') for v in ens_fc))
    diff = [int(round(ens_fc[i] - ref_fc[i])) for i in range(3)]
    signs = ['+' if d >= 0 else '' for d in diff]
    d_str = '  '.join(f'{s}{d:>8,}'.replace(',', ' ') for s, d in zip(signs, diff))
    print(f'{"":25} {"Разница (анс. - реф.)":<22} {d_str}')
    print()